# Phase 0 — Lesson 03: GPU Setup & Cloud
> Training on CPU is fine for learning. Training for real needs a GPU.

**Source:** ai-engineering-from-scratch -> Phase 0 (Setup & Tooling), Lesson 03
**Type:** Build | **Languages:** Python | **Time:** ~45 minutes
**Prerequisites:** Phase 0, Lesson 01

This lesson explains *why* GPUs accelerate AI training, *when* you need one, and *how* to estimate if your model will fit in GPU memory. Run the code cells to see visualizations of the concepts.


# Why GPUs Are Different from CPUs

---


Most lessons in phases 1-3 run fine on CPU. But once you start training CNNs, transformers, or LLMs (phases 4+), you need GPU acceleration. A training run that takes **8 hours on CPU** takes **10 minutes on GPU**.

It's not about clock speed. A CPU might run at 3.5 GHz while a GPU runs at 1.5 GHz — slower per core.
The difference is **parallelism**:

| Feature | CPU | GPU |
|---------|-----|-----|
| Cores | 8-16 powerful cores | Thousands of smaller cores |
| Best at | Complex sequential logic | Simple parallel math |
| Memory | Large system RAM | Dedicated VRAM |
| AI workload | Poor fit | Excellent fit |

Neural network training is mostly **matrix multiplication** — the same operation repeated thousands of times on different data. GPUs are built for exactly this.


# Visual: CPU vs GPU Core Architecture

---


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# CPU: few large cores
ax = axes[0]
cpu_sizes = [60, 55, 50, 45, 40]
cpu_colors = ['#34495e'] * 5
for i, s in enumerate(cpu_sizes):
    rect = plt.Rectangle((i*65, 0), s, s, facecolor=cpu_colors[i], edgecolor='white', linewidth=2)
    ax.add_patch(rect)
    ax.text(i*65 + s/2, s/2, 'Core', ha='center', va='center', color='white', fontweight='bold', fontsize=9)
ax.set_xlim(0, 330)
ax.set_ylim(0, 65)
ax.set_title('CPU: Few Powerful Cores', fontsize=11, fontweight='bold')
ax.axis('off')

# GPU: thousands of small cores
ax = axes[1]
for row in range(8):
    for col in range(10):
        rect = plt.Rectangle((col*32, row*32), 28, 28, facecolor='#e74c3c', edgecolor='white', linewidth=1)
        ax.add_patch(rect)
ax.set_xlim(0, 330)
ax.set_ylim(0, 260)
ax.set_title('GPU: Thousands of Small Cores', fontsize=11, fontweight='bold')
ax.axis('off')

plt.suptitle('Why GPUs Excel at Matrix Multiplication', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


> **Key insight:** The CPU has a few large cores for complex logic. The GPU has thousands of tiny cores for simple parallel math. Neural networks are mostly simple parallel math.


# Your Three GPU Options

---


You have three ways to access GPU power. Each has different trade-offs:

| Option | Cost | Setup | Best For |
|--------|------|-------|----------|
| 1. Local NVIDIA GPU | $0 (you own it) | Install CUDA + cuDNN | Regular use, large datasets |
| 2. Google Colab (free) | $0 | None | Quick experiments, no GPU at home |
| 3. Cloud GPU (Lambda, RunPod, Vast.ai) | $0.20–$2.00/hr | SSH + install | Serious training, large models |

Start with what you have. If you don't have a local GPU, Colab is free and requires no setup.


# Visual: Comparing Your Options

---


In [ ]:
import matplotlib.pyplot as plt

options = ['Local NVIDIA GPU', 'Google Colab (free)', 'Cloud GPU (paid)']
cost = [0, 0, 1]  # relative cost score
setup = [3, 1, 3]  # relative setup effort (1=easy, 3=hard)
flexibility = [3, 2, 3]  # relative flexibility

x = range(len(options))
width = 0.25

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width for i in x], cost, width, label='Cost', color='#e74c3c', alpha=0.8)
ax.bar(x, setup, width, label='Setup Effort', color='#f39c12', alpha=0.8)
ax.bar([i + width for i in x], flexibility, width, label='Flexibility', color='#27ae60', alpha=0.8)

ax.set_ylabel('Relative Score (0-3)', fontsize=10)
ax.set_title('GPU Options: Cost, Setup, and Flexibility', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(options, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 3.5)

plt.tight_layout()
plt.show()


> **Recommendation:** Start with Colab if you have no GPU. Move to cloud GPU when you need to train for hours. Buy a local GPU only if you'll use it regularly.


# Option 1: Local NVIDIA GPU

---


First, check if you have an NVIDIA GPU:

```bash
nvidia-smi
```

This is the **NVIDIA System Management Interface**. It shows:
- GPU utilization (%)
- Memory usage (VRAM)
- Temperature
- Running processes

Then verify PyTorch can see it:

```python
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA version: {torch.version.cuda}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
```

If `torch.cuda.is_available()` returns `False`, PyTorch was installed without CUDA support. Reinstall with:
```bash
uv pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
```


# Option 2: Google Colab (Free Tier)

---


Colab gives you a free Jupyter notebook in the cloud with a T4 GPU.

**Steps:**
1. Go to [colab.research.google.com](https://colab.research.google.com)
2. Runtime > Change runtime type > T4 GPU
3. Run `!nvidia-smi` to verify

**Then upload notebooks from this course directly to Colab.**

**Limitations:**
- Sessions timeout after ~90 minutes of inactivity
- Files don't persist between sessions (save to Google Drive or download)
- GPU may be reassigned between sessions


# Option 3: Cloud GPU (Paid)

---


For serious training, rent a GPU by the hour from Lambda Labs, RunPod, or Vast.ai:

```bash
ssh user@your-gpu-instance

pip install torch torchvision torchaudio
python -c "import torch; print(torch.cuda.get_device_name(0))"
```

You pay only for what you use. Good for multi-day training runs that would time out on Colab.


# No GPU? No Problem.

---


Most lessons work on CPU. The ones that need GPU will say so and include Colab links.

Write device-agnostic code so it works on either:

```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
```

This pattern appears throughout the course. Your code runs on GPU if available, falls back to CPU otherwise.


# Visual: CPU vs GPU Matrix Multiplication Benchmark

---


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Simulated benchmark results (representative)
matrix_sizes = [1000, 2000, 3000, 4000, 5000]
cpu_times = [0.15, 1.2, 4.1, 9.8, 19.2]  # seconds
gpu_times = [0.02, 0.05, 0.09, 0.14, 0.21]  # seconds
speedups = [c / g for c, g in zip(cpu_times, gpu_times)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Time comparison
ax = axes[0]
x = np.arange(len(matrix_sizes))
width = 0.35
ax.bar(x - width/2, cpu_times, width, label='CPU', color='#e74c3c', alpha=0.8)
ax.bar(x + width/2, gpu_times, width, label='GPU', color='#27ae60', alpha=0.8)
ax.set_xlabel('Matrix Size')
ax.set_ylabel('Time (seconds)')
ax.set_title('Matrix Multiplication Time', fontsize=11, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(matrix_sizes)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Speedup
ax = axes[1]
ax.bar(matrix_sizes, speedups, color='#3498db', alpha=0.8)
ax.set_xlabel('Matrix Size')
ax.set_ylabel('Speedup (CPU time / GPU time)')
ax.set_title('GPU Speedup Factor', fontsize=11, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

for i, s in enumerate(speedups):
    ax.text(matrix_sizes[i], s + 0.5, f'{s:.0f}x', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()


> **Note:** GPU operations are **asynchronous**. Python returns immediately while the GPU is still computing. To time GPU code accurately, you must call `torch.cuda.synchronize()` before and after the operation.


# VRAM: How Much Model Can You Fit?

---


VRAM (Video RAM) is the GPU's dedicated memory. It's separate from your system RAM and limits:
- How large a model you can load
- How big a batch you can train
- How long a sequence you can process

**Rule of thumb for fp16 (16-bit precision):** each parameter uses 2 bytes.

```
parameters = VRAM_GB * 1024^3 / 2 bytes
```

**But that's just the model weights.** You also need memory for:
- Activations (intermediate values during forward pass)
- Gradients (during backward pass)
- Optimizer states (Adam keeps 2 extra values per parameter)

In practice, you can fit about **half** the theoretical maximum for training.


# Visual: VRAM Estimation by GPU

---


In [ ]:
import matplotlib.pyplot as plt

gpus = ['GTX 1060 (6 GB)', 'RTX 3060 (12 GB)', 'RTX 3090 (24 GB)', 'A100 (40 GB)', 'H100 (80 GB)']
vram_gb = [6, 12, 24, 40, 80]

# fp16: 2 bytes per parameter. Training uses ~half for weights + activations + optimizer
params_billion_fp16 = [v * 1024**3 / 2 / 1e9 * 0.5 for v in vram_gb]
params_billion_fp32 = [v * 1024**3 / 4 / 1e9 * 0.5 for v in vram_gb]

x = range(len(gpus))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar([i - width/2 for i in x], params_billion_fp16, width, label='fp16 (training)', color='#27ae60', alpha=0.8)
ax.bar([i + width/2 for i in x], params_billion_fp32, width, label='fp32 (training)', color='#e74c3c', alpha=0.8)

ax.set_ylabel('Approx. Trainable Parameters (Billions)', fontsize=10)
ax.set_title('VRAM vs Trainable Model Size (fp16 vs fp32)', fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(gpus, rotation=15, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

for i, (f16, f32) in enumerate(zip(params_billion_fp16, params_billion_fp32)):
    ax.text(i - width/2, f16 + 0.1, f'{f16:.1f}B', ha='center', fontsize=8, fontweight='bold')
    ax.text(i + width/2, f32 + 0.1, f'{f32:.1f}B', ha='center', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.show()


> **Example:** A 24 GB GPU (RTX 3090) can fit roughly **12 billion parameters** in fp16 for training. For inference only (no gradients/optimizer), you can fit about twice that.


# Build It: Run the CPU vs GPU Benchmark Yourself

---


In [ ]:
import torch
import time

size = 5000

a_cpu = torch.randn(size, size)
b_cpu = torch.randn(size, size)

start = time.time()
c_cpu = a_cpu @ b_cpu
cpu_time = time.time() - start
print(f"CPU: {cpu_time:.3f}s")

if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    torch.cuda.synchronize()  # Wait for GPU to finish before timing
    start = time.time()
    c_gpu = a_gpu @ b_gpu
    torch.cuda.synchronize()  # Wait for GPU to finish before stopping timer
    gpu_time = time.time() - start
    print(f"GPU: {gpu_time:.3f}s")
    print(f"Speedup: {cpu_time / gpu_time:.0f}x")
else:
    print("No GPU available — CPU only")


> **Why `torch.cuda.synchronize()`?** GPU operations are asynchronous — Python starts the operation and immediately moves to the next line while the GPU is still computing. Without `synchronize()`, you'd measure only the launch time, not the actual computation time.


# Quiz Review

---


Try answering these before reading the explanations:

**Q1:** Why is a GPU faster than a CPU for training neural networks?
> **Explanation:** GPUs have thousands of cores optimized for parallel computation. Neural network training is dominated by matrix multiplications — the same math repeated on different data. GPUs execute these in parallel; CPUs execute them sequentially.

**Q2:** What does VRAM refer to?
> **Explanation:** Video RAM on the GPU, separate from system RAM. It limits model size and batch size during training.

**Q3:** What command verifies your NVIDIA GPU is detected?
> **Explanation:** `nvidia-smi` (NVIDIA System Management Interface) shows GPU utilization, memory, temperature, and running processes.

**Q4:** Why call `torch.cuda.synchronize()` when benchmarking GPU vs CPU?
> **Explanation:** GPU operations are asynchronous. Python returns immediately while the GPU computes. `synchronize()` blocks until all GPU operations finish, giving accurate timing.

**Q5:** Using the fp16 rule of thumb, how many parameters fit in 24 GB VRAM?
> **Explanation:** fp16 uses 2 bytes per parameter. 24 GB / 2 bytes = 12 billion parameters (rough estimate for weights only; training needs less due to activations and gradients).


# Practice Exercises

---


Do these in order:

**Exercise 1:** Run the benchmark above and compare CPU vs GPU times.
If you don't have a GPU, run it on Google Colab instead.

**Exercise 2:** If you don't have a GPU, open Colab, select T4 GPU runtime, run `!nvidia-smi`.

**Exercise 3:** Check your GPU memory and estimate the largest model you can fit.
Rule of thumb: 2 bytes per parameter for fp16.

```python
import torch
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    params_billion = vram_gb * 1024**3 / 2 / 1e9 * 0.5  # fp16, training estimate
    print(f"VRAM: {vram_gb:.1f} GB")
    print(f"Approx. trainable params: {params_billion:.1f} billion")
else:
    print("No GPU detected")
```


# Key Terms

---


| Term | What people say | What it actually means |
|------|----------------|----------------------|
| CUDA | "GPU programming" | NVIDIA's parallel computing platform for running code on the GPU |
| VRAM | "GPU memory" | Video RAM on the GPU, separate from system RAM. Limits model size. |
| fp16 | "Half precision" | 16-bit floating point. Uses half the memory of fp32 with minimal accuracy loss. |
| Tensor Core | "Fast matrix hardware" | Specialized GPU cores for matrix multiplication, 4-8x faster than regular cores |
| Asynchronous | "GPU runs in background" | GPU operations start and Python continues without waiting. Use `synchronize()` to wait. |


# Next Steps

---


When you've verified your GPU access (or confirmed you'll use Colab), continue to:
**Lesson 04: APIs and Keys**

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
